In [1]:
import os
import glob
import cv2
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from insightface.app import FaceAnalysis

# ================= 1. 全局配置 (CONFIGURATION) =================

# [重要] 请修改为你存放 BioVid PartA 的真实路径
ROOT_DIR = "./video"

# 输出保存路径
OUTPUT_ROOT = "./biovid"
IMAGES_DIR = os.path.join(OUTPUT_ROOT, "images")

# 图像处理参数
IMG_SIZE = 224        # OrdinalCLIP 标准输入分辨率
NUM_FRAMES = 10        # 每个视频截取 5 帧
EXPAND_RATIO = 0.25   # 人脸扩充比例

# 标签映射
LABEL_MAP = {"BL1": 0, "PA1": 1, "PA2": 2, "PA3": 3, "PA4": 4}

# 官方定义的 26 个测试集 Subject ID
OFFICIAL_TEST_SET = {
    "100914_m_39", "101114_w_37", "082315_w_60", "083114_w_55", "083109_m_60",
    "072514_m_27", "080309_m_29", "112016_m_25", "112310_m_20", 
    "092813_w_24", "112809_w_23", "112909_w_20",
    "071313_m_41", "101309_m_48", "101609_m_36", "091809_w_43", 
    "102214_w_36", "102316_w_50", "112009_w_43",
    "101814_m_58", "101908_m_61", "102309_m_61", "112209_m_51", 
    "112610_w_60", "112914_w_51", "120514_w_56"
}

# ================= 2. 功能函数 =================

def get_detector():
    """初始化 InsightFace"""
    print("--> Loading InsightFace Model (Buffalo_L)...")
    app = FaceAnalysis(name="buffalo_l", allowed_modules=['detection'])
    app.prepare(ctx_id=0, det_size=(640, 640))
    return app

def parse_filename(filename):
    stem = Path(filename).stem
    parts = stem.split('-')
    if len(parts) < 3: return None
    subject_id = parts[0]
    stimulus = parts[-2]
    if stimulus in LABEL_MAP:
        return subject_id, LABEL_MAP[stimulus], stimulus
    return None

def process_one_video(video_path, detector):
    """返回：(生成的记录列表, 状态码)
    状态码: 0=成功, 1=文件名解析失败, 2=视频太短
    """
    filename = os.path.basename(video_path)
    file_stem = Path(filename).stem
    parse_res = parse_filename(filename)
    
    if not parse_res: return [], 1 # Filename Error

    subject_id, label_int, label_str = parse_res
    
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames < 10:
        cap.release()
        return [], 2 # Too Short

    start_frame = int(total_frames * 0.2)
    frame_indices = np.linspace(start_frame, total_frames - 5, NUM_FRAMES, dtype=int)
    
    processed_records = []
    
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret: continue
        
        faces = detector.get(frame)
        if not faces: continue
        
        face = max(faces, key=lambda x: (x.bbox[2]-x.bbox[0]) * (x.bbox[3]-x.bbox[1]))
        h, w, _ = frame.shape
        b = face.bbox.astype(int)
        x1, y1, x2, y2 = b[0], b[1], b[2], b[3]
        box_w, box_h = x2 - x1, y2 - y1
        
        nx1 = max(0, int(x1 - box_w * EXPAND_RATIO))
        ny1 = max(0, int(y1 - box_h * EXPAND_RATIO))
        nx2 = min(w, int(x2 + box_w * EXPAND_RATIO))
        ny2 = min(h, int(y2 + box_h * EXPAND_RATIO))
        
        crop = frame[ny1:ny2, nx1:nx2]
        try:
            crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))
        except:
            continue
            
        save_name = f"{file_stem}_{idx}.jpg"
        save_path = os.path.join(IMAGES_DIR, save_name)
        cv2.imwrite(save_path, crop)
        
        processed_records.append({
            "path": f"images/{save_name}",
            "label": label_int,
            "subject": subject_id
        })
        
    cap.release()
    return processed_records, 0 # Success

# ================= 3. 主程序 =================

def main():
    if not os.path.exists(IMAGES_DIR):
        os.makedirs(IMAGES_DIR)

    # 1. 扫描文件
    print("--> Scanning video files...")
    video_files = glob.glob(os.path.join(ROOT_DIR, "**", "*.mp4"), recursive=True)
    total_scanned = len(video_files)
    print(f"--> Found {total_scanned} videos.")
    
    if total_scanned == 0:
        print("❌ Error: No videos found.")
        return

    # 2. 初始化模型
    detector = get_detector()

    # 3. 开始处理
    print("--> Starting Processing (Single Thread)...")
    all_data = []
    
    # 统计计数器
    stats = {
        "success_videos": 0,
        "skipped_name": 0,
        "skipped_short": 0,
        "empty_face": 0, # 视频有效但没检测到人脸
        "total_frames_extracted": 0
    }
    
    for video_path in tqdm(video_files):
        records, status = process_one_video(video_path, detector)
        
        if status == 0:
            if len(records) > 0:
                stats["success_videos"] += 1
                stats["total_frames_extracted"] += len(records)
                all_data.extend(records)
            else:
                # 视频没问题，但5帧里都没检测到人脸
                stats["empty_face"] += 1
        elif status == 1:
            stats["skipped_name"] += 1
        elif status == 2:
            stats["skipped_short"] += 1

    # 4. 生成 CSV
    df = pd.DataFrame(all_data)
    df.to_csv(os.path.join(OUTPUT_ROOT, "all_frames.csv"), index=False)
    
    # ==========================================
    # 🔍 5. 完整性自检 (Validation Check)
    # ==========================================
    print("\n" + "="*40)
    print("   📊 DATA INTEGRITY REPORT (数据体检报告)")
    print("="*40)
    
    # 理论值计算
    expected_ideal = stats["success_videos"] * NUM_FRAMES
    actual_csv_count = len(df)
    
    # 物理磁盘检查
    print("--> Verifying files on disk...")
    files_on_disk = glob.glob(os.path.join(IMAGES_DIR, "*.jpg"))
    actual_disk_count = len(files_on_disk)
    
    print(f"1. Video Statistics:")
    print(f"   - Total Scanned:     {total_scanned}")
    print(f"   - Valid Processed:   {stats['success_videos']}")
    print(f"   - Skipped (Name):    {stats['skipped_name']}")
    print(f"   - Skipped (Short):   {stats['skipped_short']}")
    print(f"   - Skipped (No Face): {stats['empty_face']}")
    
    print(f"\n2. Image Statistics:")
    print(f"   - Ideal Count (Valid * {NUM_FRAMES}): {expected_ideal}")
    print(f"   - Actual CSV Records:        {actual_csv_count}")
    print(f"   - Actual Files on Disk:      {actual_disk_count}")
    
    # 人脸检测丢失率
    if expected_ideal > 0:
        miss_count = expected_ideal - actual_csv_count
        miss_rate = (miss_count / expected_ideal) * 100
        print(f"   - Face Detection Miss Rate:  {miss_rate:.2f}% (Lost {miss_count} frames)")
    
    print(f"\n3. Consistency Check:")
    if actual_csv_count == actual_disk_count:
        print("   ✅ PASS: CSV records match disk files exactly.")
    else:
        print(f"   ❌ FAIL: Mismatch! CSV has {actual_csv_count} but Disk has {actual_disk_count}.")
        
    print("="*40 + "\n")

    # 6. 官方划分 (Official Split)
    print("--> Applying Official Hold-Out Split...")
    all_subjects = df['subject'].unique()
    test_subs_found = [s for s in all_subjects if s in OFFICIAL_TEST_SET]
    train_subs_found = [s for s in all_subjects if s not in OFFICIAL_TEST_SET]
    
    def save_txt_file(filename, subjects_list):
        subset = df[df['subject'].isin(subjects_list)]
        filepath = os.path.join(OUTPUT_ROOT, filename)
        with open(filepath, 'w') as f:
            for _, row in subset.iterrows():
                f.write(f"{row['path']} {row['label']}\n")
        print(f"    [+] {filename}: {len(subset)} images ({len(subjects_list)} subjects)")

    save_txt_file("train.txt", train_subs_found)
    save_txt_file("test.txt", test_subs_found)
    
    print(f"✅ Preprocessing Completed.")

if __name__ == "__main__":
    main()

/root/miniconda3/lib/python3.8/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


--> Scanning video files...
--> Found 8700 videos.
--> Loading InsightFace Model (Buffalo_L)...
download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:04<00:00, 68147.89KB/s] 


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_enable': '0', 'gpu_external_free': '0', 'gpu_mem_limit': '18446744073709551615', 'tunable_op_max_tuning_duration_ms': '0', 'tunable_op_tuning_enable': '0', 'cudnn_conv_use_max_workspace': '1', 'device_id': '0', 'gpu_external_alloc': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'has_user_compute_stream': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'gpu_external_empty_cache': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0'}}
model ignore: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_enable': '0', 'gpu_external_free': '0', 'gpu_mem_limit

100%|██████████| 8700/8700 [1:49:56<00:00,  1.32it/s]  



   📊 DATA INTEGRITY REPORT (数据体检报告)
--> Verifying files on disk...
1. Video Statistics:
   - Total Scanned:     8700
   - Valid Processed:   8700
   - Skipped (Name):    0
   - Skipped (Short):   0
   - Skipped (No Face): 0

2. Image Statistics:
   - Ideal Count (Valid * 10): 87000
   - Actual CSV Records:        86994
   - Actual Files on Disk:      86994
   - Face Detection Miss Rate:  0.01% (Lost 6 frames)

3. Consistency Check:
   ✅ PASS: CSV records match disk files exactly.

--> Applying Official Hold-Out Split...
    [+] train.txt: 60994 images (61 subjects)
    [+] test.txt: 26000 images (26 subjects)
✅ Preprocessing Completed.
